In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [2]:
from icet.tools.structure_generation import generate_sqs_from_supercells
from icet import ClusterSpace
from ase import Atoms
from ase.io import read
from ase.visualize import view
from spglib import find_primitive
from ase.build import sort

In [3]:
atoms = read('HgTe.cif')

In [4]:
hgte_prim = find_primitive((atoms.cell, atoms.get_scaled_positions(), atoms.numbers))
hgte_prim = Atoms(cell=hgte_prim[0], scaled_positions=hgte_prim[1], numbers=hgte_prim[2], pbc=True)

In [5]:
 cs = ClusterSpace(hgte_prim, [12, 8], [['Hg', 'Cd'], ['Te']])

In [6]:
print(cs)

====================================== Cluster Space ======================================
 space group                            : F-43m (216)
 chemical species                       : ['Cd', 'Hg'] (sublattice A)
 cutoffs                                : 12.0000 8.0000
 total number of parameters             : 11
 number of parameters by order          : 0= 1  1= 1  2= 6  3= 3
 fractional_position_tolerance          : 2e-06
 position_tolerance                     : 1e-05
 symprec                                : 1e-05
-------------------------------------------------------------------------------------------
index | order |  radius  | multiplicity | orbit_index | multicomponent_vector | sublattices
-------------------------------------------------------------------------------------------
   0  |   0   |   0.0000 |        1     |      -1     |           .           |      .     
   1  |   1   |   0.0000 |        1     |       0     |          [0]          |      A     
   2  |   2  

In [10]:
from ase.build.supercells import find_optimal_cell_shape, make_supercell

In [20]:
trans_mat = find_optimal_cell_shape(hgte_prim.cell, 160, 'sc')

scell = make_supercell(hgte_prim, trans_mat)

scell.cell.cellpar()

array([23.72441349, 21.82328635, 23.26370141, 99.82041355, 85.50072655,
       75.47084567])

In [21]:
mct = generate_sqs_from_supercells(cs, [scell], {'Hg': 0.80, 'Cd': 0.20}, n_steps=10000)

In [22]:
mct = sort(mct)

In [23]:
from aiida import orm

In [26]:
node = orm.StructureData(ase=mct)
node.label = 'Hg0.80Cd0.20Te 320 ATOMS'
node.descriptioin = 'MCT structure generated by ICET with [12, 8] cut off radii with 320 supercell size'

In [27]:
node.store()

<StructureData: uuid: 2aaedf82-15c3-4851-8d8b-7561aa997a0d (pk: 670871)>

In [28]:
from aiida_grouppathx import GroupPathX

In [29]:
basepath = GroupPathX('mct-defect')['sqs']['structure_hg_0_8_320_atoms'] = node

In [30]:
view(node.get_ase(), viewer='weas')

WeasWidget(children=(BaseWidget(atoms={'species': {'Cd': 'Cd', 'Hg': 'Hg', 'Te': 'Te'}, 'cell': [19.73990523, …